This notebook will conduct the preprocessing of the text data collected for the Kitwe project. The motivation is to uset this text data to classify the document as fake and genuine. In order to have a better understanding of the dataset we will try the following methods:

1. Tokenization
2. lowercasing
3. Converting date and time to a standard format
4. lemmatization
5. removing duplicate entries
6. removing punctuations (.,:-_()[]?''""!)

We have collected a bunch of news data using RSS feeds. Now we will take a look at the data and do some cleaning with pandas and spacy.

In [1]:
#import regex
import spacy
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Read the raw text data for clearning
df = pd.read_csv('../../data/raw-new.csv')
df.head(10)

,cd ..Source,Category,Headline,Link,Description,Date,Author
0,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: GDP – Legal,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,"Wed, 20 Nov 2024 16:31:18 +0000",Lovejoy Musundire
1,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Assurance Specialist –...,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,"Wed, 20 Nov 2024 15:15:28 +0000",Lovejoy Musundire
2,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Internal Auditor – In...,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,"Wed, 20 Nov 2024 15:14:39 +0000",Lovejoy Musundire
3,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Engineer – Protection,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,"Wed, 20 Nov 2024 15:13:41 +0000",Lovejoy Musundire
4,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Mechanic II,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,"Wed, 20 Nov 2024 15:13:05 +0000",Lovejoy Musundire
5,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Technician – Electrical,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,"Tue, 24 Sep 2024 09:10:46 +0000",Lovejoy Musundire
6,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Parks & Gardens Assistant,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,"Tue, 24 Sep 2024 09:09:59 +0000",Lovejoy Musundire
7,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Technician – Mechanical,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,"Tue, 24 Sep 2024 09:09:22 +0000",Lovejoy Musundire
8,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: CAPEX Engineer,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,"Mon, 23 Sep 2024 15:29:42 +0000",Lovejoy Musundire
9,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Engineer – SCADA,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,"Mon, 23 Sep 2024 15:29:32 +0000",Lovejoy Musundire


For text classification, the important columns are the headline and description. The source of the data and author information are important for classifying if the news is fake or not. For the moment, we can defnitely get rid of the link column. We will also convert the date to standard pandas date format so that we can get an idea of when the news was published.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226 entries, 0 to 2225
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   cd ..Source  2226 non-null   object
 1   Category     2226 non-null   object
 2   Headline     2226 non-null   object
 3   Link         2226 non-null   object
 4   Description  2220 non-null   object
 5   Date         2226 non-null   object
 6   Author       2226 non-null   object
dtypes: object(7)
memory usage: 121.9+ KB


In [4]:
#df.drop(columns=['Link'], inplace=True)

In [5]:
df.rename(columns={'cd ..Source':'Source'}, inplace=True)

In [6]:
df.columns

Index(['Source', 'Category', 'Headline', 'Link', 'Description', 'Date',
       'Author'],
      dtype='object')

In [7]:
# converting the date columnn to pandas date and time format
df['Date'] = pd.to_datetime(df['Date'])
df['Date'].iloc[:5]

0   2024-11-20 16:31:18+00:00
1   2024-11-20 15:15:28+00:00
2   2024-11-20 15:14:39+00:00
3   2024-11-20 15:13:41+00:00
4   2024-11-20 15:13:05+00:00
Name: Date, dtype: datetime64[ns, UTC]

In [8]:
# let's check for Nan values and duplicate entries
df.isna().sum()

Source         0
Category       0
Headline       0
Link           0
Description    6
Date           0
Author         0
dtype: int64

Okay. So there are no null values. 

In [9]:
# Look for duplicated entries
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
2221     True
2222     True
2223     True
2224     True
2225     True
Length: 2226, dtype: bool

In [10]:
# Let's drop the duplicates
df.drop_duplicates(inplace=True)
df.shape

(1936, 7)

In [11]:
# Let's take a look at the unique sources, caegories of the news
df['Source'].value_counts()

Source
Mwebantu                   300
Lusaka Times               300
Kitwe Online               300
Zambia Monitor             300
Zambian Eye                300
Copperbelt Energy          225
Daily Revelation Zambia    113
News Invasion 24            43
ZNBC                        41
DailyMail                   10
Tech Africa News             4
Name: count, dtype: int64

In [12]:
# Look at the unique categories
df['Category'].unique()[:50]

array(['Careers,Current Careers',
       'Corporate Announcements,Downloads,Featured,Green Bond', 'Careers',
       'Corporate Social Responsibility,Featured',
       'Corporate Announcements,Featured', 'Featured,News',
       'Annual Report,Corporate Announcements,Downloads,Featured,AF',
       'Corporate Announcements,Featured,AF',
       'Corporate Announcements,Downloads,Featured,AF', 'News',
       'Corporate Social Responsibility,Featured,News,AF',
       'Corporate Social Responsibility,News',
       'Corporate Social Responsibility,Featured,News,Power Dynamos',
       'Corporate Social Responsibility,Power Dynamos',
       'Downloads,Featured,News', 'Careers,Featured,Power Dynamos',
       'Corporate Social Responsibility,News,Power Dynamos',
       'Corporate Announcements,Downloads,Featured', 'Power Dynamos',
       'Corporate Social Responsibility,Featured,News',
       'Annual Report,Corporate Announcements,Downloads,Featured',
       'Featured,News,Project', 'Corporate Ann

It looks like the Category column consists of many entries and often mixed with headline column for many of the training data. Therefore it is important to come up with a unique set of categories and appending that information to the Category column.

In [13]:
# Defining the major categories of news here after looking into different possibilities. We will now categorize the data based on this list
cat_list = ['education', 'politics', 'local news', 'development', 'narcotics',
       'economy news', 'business news', 'health and wellness', 'sports',
       'entertainment', 'health', 'fashion', 'tourism', 'crime',
       'agriculture', 'environment', 'business', 'technology', 'science']

### Text Preprocessing with Spacy

In [14]:
# Loading the spacy english language model
# This model was trained on large corpus of labeled text data. Labels like sentence parsing, POS tagging, named entity recongnition 
nlp = spacy.load('en_core_web_sm')

In [15]:
# Try out spacy in a single text before the whole training examples
# Now we will go through the description of the training example for preprocessing
s = df['Description'].iloc[0]
s

'We currently have career opportunities in the following field VAC-2024-0026: GDP LEGAL Grade: CEC GDP | Contract Type: Fixed Term | Location: Kitwe Are you a fresh graduate with a Bachelor of Law Degree (LLB) and an Advocate of the High Court of Zambia; seeking to acquire real-world, hands-on experience to hone your strengths and'

In [16]:
doc = nlp(s)

# Getting tokens and their indexes
tok = [(t.i, t.text) for t in doc]
tok

[(0, 'We'),
 (1, 'currently'),
 (2, 'have'),
 (3, 'career'),
 (4, 'opportunities'),
 (5, 'in'),
 (6, 'the'),
 (7, 'following'),
 (8, 'field'),
 (9, 'VAC-2024'),
 (10, '-'),
 (11, '0026'),
 (12, ':'),
 (13, 'GDP'),
 (14, 'LEGAL'),
 (15, 'Grade'),
 (16, ':'),
 (17, 'CEC'),
 (18, 'GDP'),
 (19, '|'),
 (20, 'Contract'),
 (21, 'Type'),
 (22, ':'),
 (23, 'Fixed'),
 (24, 'Term'),
 (25, '|'),
 (26, 'Location'),
 (27, ':'),
 (28, 'Kitwe'),
 (29, 'Are'),
 (30, 'you'),
 (31, 'a'),
 (32, 'fresh'),
 (33, 'graduate'),
 (34, 'with'),
 (35, 'a'),
 (36, 'Bachelor'),
 (37, 'of'),
 (38, 'Law'),
 (39, 'Degree'),
 (40, '('),
 (41, 'LLB'),
 (42, ')'),
 (43, 'and'),
 (44, 'an'),
 (45, 'Advocate'),
 (46, 'of'),
 (47, 'the'),
 (48, 'High'),
 (49, 'Court'),
 (50, 'of'),
 (51, 'Zambia'),
 (52, ';'),
 (53, 'seeking'),
 (54, 'to'),
 (55, 'acquire'),
 (56, 'real'),
 (57, '-'),
 (58, 'world'),
 (59, ','),
 (60, 'hands'),
 (61, '-'),
 (62, 'on'),
 (63, 'experience'),
 (64, 'to'),
 (65, 'hone'),
 (66, 'your'),
 (67, 's

In [17]:
# Getting each sentences as well
print([sent for sent in doc.sents])

[We currently have career opportunities in the following field VAC-2024-0026: GDP LEGAL Grade: CEC GDP | Contract Type: Fixed Term | Location:, Kitwe Are you a fresh graduate with a Bachelor of Law Degree (LLB) and an Advocate of the High Court of Zambia; seeking to acquire real-world, hands-on experience to hone your strengths and]


In [18]:
# case folding
print([t.lower_ for t in doc])

['we', 'currently', 'have', 'career', 'opportunities', 'in', 'the', 'following', 'field', 'vac-2024', '-', '0026', ':', 'gdp', 'legal', 'grade', ':', 'cec', 'gdp', '|', 'contract', 'type', ':', 'fixed', 'term', '|', 'location', ':', 'kitwe', 'are', 'you', 'a', 'fresh', 'graduate', 'with', 'a', 'bachelor', 'of', 'law', 'degree', '(', 'llb', ')', 'and', 'an', 'advocate', 'of', 'the', 'high', 'court', 'of', 'zambia', ';', 'seeking', 'to', 'acquire', 'real', '-', 'world', ',', 'hands', '-', 'on', 'experience', 'to', 'hone', 'your', 'strengths', 'and']


In [19]:
# stop word removal
# Let's look at the spacy's default list of stop word list
print(nlp.Defaults.stop_words)

{'anyhow', 'often', 'where', 'well', 'around', 'the', 'nothing', 'always', 'me', 'whole', "'s", 'eleven', 'here', 'we', 'very', 'else', 'on', 'besides', 'are', 'those', 'hereby', 'my', 'whereafter', 'whence', 'alone', 'moreover', 'against', 'only', 'who', 'your', 'amongst', 'am', 'you', 'about', 'in', 'therein', 'since', 'be', 'anything', 'do', 'seem', 'off', 'would', 'therefore', 'afterwards', 'herself', 'still', 'among', 'of', 'for', 'have', 'make', 'below', 'move', 'perhaps', 'under', 'ca', 'ever', 'whereby', 'an', 'somewhere', 'get', 'neither', 'toward', 'go', 'latter', 'out', 'few', 'as', 'everywhere', 'if', '‘s', 'while', 'whither', 'that', 'she', 'yet', '’re', 'nowhere', 'or', 'cannot', 'although', 'least', 'whatever', 'sometime', 'six', 'did', 'almost', 'hence', 'could', 'say', 'meanwhile', 'top', 'so', 'thru', 'some', 'whom', 'name', 'everyone', 'within', 'see', 'is', 'between', 'every', 'never', '’m', 'whenever', 'now', 'they', 'done', 'together', 'none', 'yours', 'during', '

In [20]:
print(len(nlp.Defaults.stop_words))

326


Interstingly, there are some words in the list which I won't consider to be stopping words.

In [21]:
# Remove the common occuring stop words here
# Remove punctuations as well
# NOw look at tokens which are not in the list
print([t.text for t in doc if (not t.is_stop) and (not t.is_punct)])

['currently', 'career', 'opportunities', 'following', 'field', 'VAC-2024', '0026', 'GDP', 'LEGAL', 'Grade', 'CEC', 'GDP', '|', 'Contract', 'Type', 'Fixed', 'Term', '|', 'Location', 'Kitwe', 'fresh', 'graduate', 'Bachelor', 'Law', 'Degree', 'LLB', 'Advocate', 'High', 'Court', 'Zambia', 'seeking', 'acquire', 'real', 'world', 'hands', 'experience', 'hone', 'strengths']


### Cleaning of the data

In [22]:
def preprocess_text(text):
    """
    Tokenization, lemmatization, case folding, removal of stop words and
    punctuations
    """
    doc = nlp(text)
    return [tok.lemma_.lower() for tok in doc if (tok.is_alpha) and (not tok.is_stop) and (not tok.is_punct) and (not tok.is_space)]

print(preprocess_text(s))

['currently', 'career', 'opportunity', 'follow', 'field', 'gdp', 'legal', 'grade', 'cec', 'gdp', 'contract', 'type', 'fix', 'term', 'location', 'kitwe', 'fresh', 'graduate', 'bachelor', 'law', 'degree', 'llb', 'advocate', 'high', 'court', 'zambia', 'seek', 'acquire', 'real', 'world', 'hand', 'experience', 'hone', 'strength']


In [23]:
# Let's make a copy of the dataset before doing the preprocessing
df_clean = df.copy()
# Now let's apply this function on the headline and the description column
df_clean['Headline'] = df_clean['Headline'].apply(preprocess_text)

In [25]:
df_clean.head(5)

,Source,Category,Headline,Link,Description,Date,Author
0,Copperbelt Energy,"Careers,Current Careers","[cec, career, opportunity, gdp, legal]",https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 16:31:18+00:00,Lovejoy Musundire
1,Copperbelt Energy,"Careers,Current Careers","[cec, career, opportunity, assurance, speciali...",https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:15:28+00:00,Lovejoy Musundire
2,Copperbelt Energy,"Careers,Current Careers","[cec, career, opportunity, internal, auditor, ...",https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:14:39+00:00,Lovejoy Musundire
3,Copperbelt Energy,"Careers,Current Careers","[cec, career, opportunity, engineer, protection]",https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:13:41+00:00,Lovejoy Musundire
4,Copperbelt Energy,"Careers,Current Careers","[cec, career, opportunity, mechanic, ii]",https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:13:05+00:00,Lovejoy Musundire


In [26]:
df_clean['Description'] = df_clean['Description'].astype(str).apply(preprocess_text)
df_clean.head(5)

,Source,Category,Headline,Link,Description,Date,Author
0,Copperbelt Energy,"Careers,Current Careers","[cec, career, opportunity, gdp, legal]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 16:31:18+00:00,Lovejoy Musundire
1,Copperbelt Energy,"Careers,Current Careers","[cec, career, opportunity, assurance, speciali...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:15:28+00:00,Lovejoy Musundire
2,Copperbelt Energy,"Careers,Current Careers","[cec, career, opportunity, internal, auditor, ...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:14:39+00:00,Lovejoy Musundire
3,Copperbelt Energy,"Careers,Current Careers","[cec, career, opportunity, engineer, protection]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:13:41+00:00,Lovejoy Musundire
4,Copperbelt Energy,"Careers,Current Careers","[cec, career, opportunity, mechanic, ii]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:13:05+00:00,Lovejoy Musundire


### Label annotation
Now since we have finished the data cleaning process, the next important thing is to annotate the data as real or fake. We need to come up with a clear strategy of how to do that.
1. Checking if the source is a reliable and reputable news channel
2. Check if the news domain is suspicious
3. Look for clickbaits which has exaggerated use of sensational keywords
4. Check if the headline and description matches each other
5. check the polarity of the headline or the description, if its too negative
6. Check for execessive capitalization
7. Check for vague authors
8. Check for suspicious links

Let's classify the news as fake if it satisfies atleast 2 of these criterias.

In [ ]:
class FakeNewsDetector():
    """
    Class to detect the genuinity of news elements
    within a given dataframe
    """

    def __init__(self, df):
        self.df = df # pandas data frame
        self.zambian_reputable_sources = ['daily-mail.co.zm', 'times.co.zm', 'znbc.co.zm', 'flavaradioandtv.com', 
            'lusakatimes.com', 'kitwetimes.com','zambiamonitor.com']
        self.suspicious_domain_pattern = re.compile(r'\\.(info|lo|ru|cn|xyz|top|news|live|buzz|click|online)$')
        
        # list of sensational words
        self.sensational_keywords = [
            'shocking', 'unbelievable', 'amazing', 'incredible', 'secret', 
            'exposed', 'you won’t believe', 'scandal', 'controversy'
        ]
    def check_vague_author(self, author):
        """
        Checking for vague authors
        """
        vague_authors = ['admin', 'editor', 'newsroom', 'staff', 'unknown']
        if author.lower() in vague_authors:
            return True
        else:
            return Falset
    def similarity_head_desc(self, head, desc):
        """
        Try to find a cosine similarity between headline and
        description using tf-idf
        """
        comb_text = [head, desc]
        tfidf_vectorizer = TfidfVectorizer()
        # This step will basically create a unique vocabulary and a matrix with features for each corpuse
        # elements. Or basically create a tf-idf vector for each row based on how frequency a token appears in one document and how the same 
        # appear in other documents
        tfidf_mat = tfidf_vectorizer.fit_transform(comb_text)
        sim_score = cosine_similarity(tfidf_mat[0,:], tfidf_mat[1,:])
        
    
        
        

In [27]:
head, desc = df.loc[100, ['Headline', 'Description']]
head
comb_txt = [head, desc]
print(comb_txt)

['“I will not put you to shame” – new Power coach', 'Newly appointed Power Dynamos Football Club (“Power”) head coach, Mr. Guston Mutobo, has assured the club’s sponsor, Copperbelt Energy Corporation Plc (CEC) and its supporters, that he would not put them to shame but would work hard to turn around the Club’s dwindling fortunes. Mutobo spoke during his introduction to CEC and club supporters’ representatives']


In [29]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_mat = tfidf_vectorizer.fit_transform(comb_txt)
print(tfidf_mat.shape)
sim_score = cosine_similarity(tfidf_mat[0,:], tfidf_mat[1,:])
print(sim_score)

(2, 46)
[[0.20914469]]
